# YOLOv12n x VisDrone -- `baseline` Notebook

> YOLOv12n chuan, khong modification. Reference mAP va FPS.
> T4 GPU | ~2-3h

In [1]:
!git clone https://github.com/trong5nhan6/yolov12n-visdrone.git
%cd yolov12n-visdrone

Cloning into 'yolov12n-visdrone'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 65 (delta 21), reused 57 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 74.46 KiB | 2.07 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/yolov12n-visdrone


In [2]:
!mkdir -p datasets
!gdown 1_DTApN9Vwhex4v2m_39aTmNVF4wUA5E- -O datasets/VisDrone.zip
!unzip -q datasets/VisDrone.zip -d datasets

Downloading...
From (original): https://drive.google.com/uc?id=1_DTApN9Vwhex4v2m_39aTmNVF4wUA5E-
From (redirected): https://drive.google.com/uc?id=1_DTApN9Vwhex4v2m_39aTmNVF4wUA5E-&confirm=t&uuid=504de60e-2294-45a5-b6f6-179d7f55c851
To: /content/yolov12n-visdrone/datasets/VisDrone.zip
100% 1.94G/1.94G [00:22<00:00, 86.5MB/s]


## 1. Kiem tra GPU

In [3]:
import subprocess, torch
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode==0 else 'No GPU')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Thu May 28 15:51:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Cai dat dependencies

In [4]:
%pip install -q ultralytics>=8.3.0 omegaconf>=2.3.0 rich>=13.0.0 thop
import ultralytics; ultralytics.checks()
print(f'ultralytics: {ultralytics.__version__}')

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 50.8/112.6 GB disk)
ultralytics: 8.4.56


## 4. Upload project code

Chon **mot trong 3 cach**:

- Cach 1: Upload file `.zip` tu may tinh
- Cach 2: Copy tu Google Drive
- Cach 3: `git clone` tu GitHub

In [5]:
import os, shutil, zipfile
PROJECT_DIR = '/content/yolov12n-visdrone'

# -- Cach 1: Upload zip -------------------------------------------------
# from google.colab import files
# up = files.upload()
# with zipfile.ZipFile(list(up.keys())[0]) as z: z.extractall('/content/')

# -- Cach 2: Tu Drive ---------------------------------------------------
# shutil.copytree('/content/drive/MyDrive/yolov12n-visdrone-code',
#                 PROJECT_DIR, dirs_exist_ok=True)

# -- Cach 3: Git clone --------------------------------------------------
# !git clone https://github.com/your-username/yolov12n-visdrone {PROJECT_DIR}

print('OK' if os.path.exists(PROJECT_DIR) else 'Project chua duoc upload')

OK


In [6]:
import os
os.chdir('/content/yolov12n-visdrone')
print(os.getcwd(), os.listdir('.'))

/content/yolov12n-visdrone ['.gitignore', 'requirements.txt', '=13.0.0', 'val.py', 'utils', 'configs', 'notebooks', '.git', 'datasets', 'predict.py', 'train.py', 'models', '=8.3.0', '=2.3.0', 'scripts', 'README.md']


## 5. Load VisDrone dataset tu Google Drive

> Dataset da convert sang YOLO format roi, zip thanh `VisDrone.zip` va upload Drive.
> Khong can chay `prepare_visdrone.py` lai.

Thu tu: (1) VisDrone.zip tren Drive -> giai nen; (2) Thu muc VisDrone/ -> symlink; (3) Bao loi.

In [7]:
!python scripts/check_dataset.py --dataset-root datasets/VisDrone

2026-05-28 15:51:26,654 | INFO     | Checking dataset at: datasets/VisDrone
2026-05-28 15:51:32,721 | INFO     | 
──────────────────────────────────────────────────
2026-05-28 15:51:32,723 | INFO     |   Split: TRAIN
2026-05-28 15:51:32,723 | INFO     | ──────────────────────────────────────────────────
2026-05-28 15:51:32,723 | INFO     |   Images total   :   6471
2026-05-28 15:51:32,723 | INFO     |   Images paired  :   6471
2026-05-28 15:51:32,723 | INFO     |   Missing labels :      0
2026-05-28 15:51:32,723 | INFO     |   Empty labels   :      0
2026-05-28 15:51:32,723 | INFO     |   Label errors   :      0
2026-05-28 15:51:32,723 | INFO     |   Objects total  : 343204
2026-05-28 15:51:32,723 | INFO     | 
  Size distribution (diagonal pixels):
    Tiny   (<32px) : 131243 (38.2%)
    Small (32-96px): 162269 (47.3%)
    Medium (>96px) :  49692 (14.5%)
2026-05-28 15:51:32,723 | INFO     | 
  Class distribution:
2026-05-28 15:51:32,723 | INFO     |     pedestrian          :  79337  █

## 6. Cau hinh training

In [8]:
import yaml
with open('configs/base.yaml') as f: base=yaml.safe_load(f)
try:
    with open('configs/baseline.yaml') as f: idea=yaml.safe_load(f)
    print('Config: configs/baseline.yaml')
except FileNotFoundError: idea={}; print('Base config only')
merged={**base,**idea}
[print(' ',k,merged.get(k,'N/A')) for k in ['epochs','batch','imgsz','lr0']]


Config: configs/baseline.yaml
  epochs 100
  batch N/A
  imgsz N/A
  lr0 0.01


[None, None, None, None]

In [9]:
IDEA='cagi'; MODEL='yolov12n'
EPOCHS=6; BATCH=32; IMGSZ=640; DEVICE='0'
TEST_EVERY=3; PROJECT_OUT='runs/detect/runs/train'
EXP_NAME=MODEL+'_'+IDEA
print(MODEL,IDEA,EPOCHS,'epochs -> '+PROJECT_OUT+'/'+EXP_NAME)


yolov12n cagi 6 epochs -> runs/detect/runs/train/yolov12n_cagi


## 7. Training

- Val tu dong sau moi epoch (ultralytics val=True)
- Test + full metrics report sau moi TEST_EVERY epochs

In [10]:
import os; os.makedirs('logs', exist_ok=True)
!python train.py \
    --model      {MODEL} \
    --idea       {IDEA} \
    --epochs     {EPOCHS} \
    --batch      {BATCH} \
    --imgsz      {IMGSZ} \
    --device     {DEVICE} \
    --project    {PROJECT_OUT} \
    --name       {EXP_NAME} \
    --test-every {TEST_EVERY} \
    --log-file   logs/{EXP_NAME}.log

2026-05-28 15:51:35 | INFO     | __main__ | ============================================================
2026-05-28 15:51:35 | INFO     | __main__ |   YOLOv12n VisDrone Trainer
2026-05-28 15:51:35 | INFO     | __main__ |   model=yolov12n  idea=cagi
2026-05-28 15:51:35 | INFO     | __main__ | ============================================================
2026-05-28 15:51:35 | INFO     | __main__ | Merged experiment config: configs/idea_cagi.yaml
2026-05-28 15:51:35 | INFO     | __main__ | Config:
model: yolov12n
idea: cagi
pretrained: yolo12n.pt
data: configs/visdrone.yaml
img_size: 640
epochs: 6
batch_size: 16
workers: 8
optimizer: SGD
lr0: 0.01
lrf: 0.01
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 3
warmup_momentum: 0.8
warmup_bias_lr: 0.1
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 0.0
translate: 0.2
scale: 0.9
shear: 0.0
perspective: 0.0
flipud: 0.5
fliplr: 0.5
mosaic: 1.0
mixup: 0.1
copy_paste: 0.3
close_mosaic: 20
val_split: val
test_split: test
test_every_n_epochs: 3
save_p

## 8. Danh gia sau training -- Full Metrics Report

3 phan:
1. **Accuracy**: mAP50, mAP50-95, Precision, Recall, AP per-class
2. **Efficiency**: GFLOPs, Params, Model size, Latency, FPS
3. **Efficiency Ratios**: mAP50/GFLOPs, mAP50/Params, mAP50*FPS

In [11]:
import os
WEIGHTS = f'{PROJECT_OUT}/{EXP_NAME}/weights/best.pt'
if not os.path.exists(WEIGHTS):
    WEIGHTS = f'{PROJECT_OUT}/{EXP_NAME}/weights/last.pt'
    print(f'best.pt not found, using: {WEIGHTS}')
else:
    print(f'Weights: {WEIGHTS}')

best.pt not found, using: runs/detect/runs/train/yolov12n_cagi/weights/last.pt


In [12]:
WEIGHTS = "/content/yolov12n-visdrone/runs/detect/runs/detect/runs/train/yolov12n_baseline/weights/best.pt"

In [13]:
# -- Val split ---------------------------------------------------------
!python val.py \
    --weights        {WEIGHTS} \
    --data           configs/visdrone.yaml \
    --split          val \
    --imgsz          {IMGSZ} \
    --device         {DEVICE} \
    --model-name     {MODEL} \
    --idea           {IDEA} \
    --save-csv       runs/metrics.csv \
    --benchmark-runs 30

2026-05-28 16:12:18 | ERROR    | Weights file not found: /content/yolov12n-visdrone/runs/detect/runs/detect/runs/train/yolov12n_baseline/weights/best.pt


In [14]:
# -- Test split --------------------------------------------------------
!python val.py \
    --weights        {WEIGHTS} \
    --data           configs/visdrone.yaml \
    --split          test \
    --imgsz          {IMGSZ} \
    --device         {DEVICE} \
    --model-name     {MODEL} \
    --idea           {IDEA} \
    --save-csv       runs/metrics.csv \
    --benchmark-runs 30

2026-05-28 16:12:19 | ERROR    | Weights file not found: /content/yolov12n-visdrone/runs/detect/runs/detect/runs/train/yolov12n_baseline/weights/best.pt


In [15]:
import pandas as pd, os
if os.path.exists('runs/metrics.csv'):
    df = pd.read_csv('runs/metrics.csv')
    want = ['timestamp','model','idea','split',
            'acc_map50','acc_map50_95','acc_precision','acc_recall',
            'eff_gflops','eff_params_m','eff_fps','eff_latency_ms',
            'ratio_map50_per_gflop','ratio_map50_x_fps']
    cols = [c for c in want if c in df.columns]
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 200)
    print(df[cols].to_string(index=False))
else:
    print('metrics.csv not found')

metrics.csv not found


## 9. Visualize ket qua training

In [16]:
from IPython.display import Image, display
import os
exp_dir = f'{PROJECT_OUT}/{EXP_NAME}'
for name in ['results.png','confusion_matrix.png','PR_curve.png','F1_curve.png']:
    p = os.path.join(exp_dir, name)
    if os.path.exists(p):
        print(name); display(Image(p, width=820))

In [17]:
import pandas as pd, os
rcsv = f'{PROJECT_OUT}/{EXP_NAME}/results.csv'
if os.path.exists(rcsv):
    df = pd.read_csv(rcsv); df.columns = df.columns.str.strip()
    col = 'metrics/mAP50(B)'
    if col in df.columns:
        best = df.loc[df[col].idxmax()]
        print(f'Best epoch : {int(best.get("epoch",0))}')
        print(f'mAP50      : {best[col]:.4f}')
        print(f'mAP50-95   : {best.get("metrics/mAP50-95(B)",0):.4f}')
        print(f'Precision  : {best.get("metrics/precision(B)",0):.4f}')
        print(f'Recall     : {best.get("metrics/recall(B)",0):.4f}')
    else:
        print(df.tail(3).to_string())

## 11. Demo Inference (tuy chon)

In [18]:
import glob, random
imgs = glob.glob('datasets/VisDrone/images/test/*.jpg')
if imgs:
    s = random.choice(imgs)
    print('Demo:', s)
    os.system(f'python predict.py --weights {WEIGHTS} --source "{s}"'
              f' --conf 0.25 --imgsz {IMGSZ} --device {DEVICE}'
              f' --project runs/predict --name demo_{IDEA}')

Demo: datasets/VisDrone/images/test/9999941_00000_d_0000007.jpg


In [19]:
from IPython.display import Image, display
import glob
for p in glob.glob(f'runs/predict/demo_{IDEA}/*.jpg')[:3]:
    display(Image(p, width=820))

In [20]:
import glob
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO

# load model
model = YOLO(WEIGHTS)

# lấy nhiều ảnh random
imgs = glob.glob('datasets/VisDrone/images/test/*.jpg')
samples = random.sample(imgs, 3)   # số ảnh muốn predict

# predict batch
results = model.predict(
    source=samples,
    conf=0.25,
    imgsz=IMGSZ,
    device=DEVICE,
    save=True,
    project='runs/predict',
    name=f'batch_{IDEA}'
)

# plot
plt.figure(figsize=(18, 12))

for i, r in enumerate(results):
    img = r.plot()[:, :, ::-1]  # BGR -> RGB

    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(samples[i].split('/')[-1], fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/yolov12n-visdrone/runs/detect/runs/detect/runs/train/yolov12n_baseline/weights/best.pt'